# 레슨 02 — NumPy 다차원 배열과 시뮬레이션 정답지

> 교사·관리자 전용. 학생에게 배포하지 않는다.

이 정답지는 학생용 `mission.md` 의 문제 1~15와 번호가 1:1로 대응한다. 각 정답에는 코드와 함께 `왜 이 코드가 정답인지` 설명을 포함한다. 학생 답안은 줄 단위로 같을 필요는 없지만, 배열의 모양과 축 방향, 최종 계산값이 맞아야 한다.

## 환경 셀

In [ ]:
import os
import numpy as np

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
if IS_COLAB:
    DATA_BASE = "https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-data-analysis/lectures/02/data"
else:
    DATA_BASE = "./data"
np.random.seed(42)

print("numpy:", np.__version__)
print("data base:", DATA_BASE)

---

## 문제 1 정답 — 점수 데이터 불러오기

In [ ]:
raw = np.loadtxt(f"{DATA_BASE}/student_scores_2d.csv", delimiter=",", skiprows=1)

print("처음 5행:\n", raw[:5])
print("shape:", raw.shape)
print("ndim:", raw.ndim)
print("size:", raw.size)

rows, cols = raw.shape
print(f"이 데이터는 {rows}명, {cols}개 컬럼 구조입니다.")

### 왜 이 코드가 정답인지

`student_scores_2d.csv` 는 첫 줄에 컬럼명이 있으므로 `skiprows=1` 로 헤더를 제외해야 숫자 배열로 읽힌다. `np.loadtxt` 는 기본적으로 2차원 배열을 만들며, 이 데이터는 120행 5열이다. `shape` 는 행과 열 개수를, `ndim` 은 차원 수를, `size` 는 전체 원소 수를 보여준다. 처음부터 이 세 값을 확인해야 뒤 문제에서 슬라이싱 방향을 틀리지 않는다.

**채점 기준**

| 항목 | 통과 기준 |
|---|---|
| CSV 로드 | `delimiter=","`, `skiprows=1` 사용 |
| 구조 확인 | `shape`, `ndim`, `size` 중 3개 모두 출력 |
| 해석 | 120명, 5개 컬럼 구조라고 말할 수 있음 |

---

## 문제 2 정답 — 점수 열만 슬라이싱하기

In [ ]:
class_id = raw[:, 0].astype(int)
student_id = raw[:, 1].astype(int)
scores = raw[:, 2:]

print("class_id shape:", class_id.shape)
print("student_id shape:", student_id.shape)
print("scores shape:", scores.shape)
print("앞 3명 점수:\n", scores[:3])

### 왜 이 코드가 정답인지

CSV 컬럼 순서는 `class_id`, `student_id`, `math`, `english`, `science` 다. 따라서 `raw[:, 0]` 은 반 번호, `raw[:, 1]` 은 학생 번호, `raw[:, 2:]` 는 세 과목 점수만 선택한다. 쉼표 앞의 `:` 는 모든 행을 의미하고, 쉼표 뒤의 `2:` 는 세 번째 열부터 끝까지를 의미한다. `scores.shape` 가 `(120, 3)` 이면 120명과 세 과목만 남았다는 뜻이다.

**자주 틀리는 답안**

| 답안 | 문제 |
|---|---|
| `raw[2:]` | 행 2부터 끝까지를 선택해 첫 두 학생이 사라짐 |
| `raw[:, 2:4]` | science 열이 빠짐 |
| `raw[:, 0:3]` | class_id, student_id 가 점수에 섞임 |

---

## 문제 3 정답 — 과목별 평균과 표준편차

In [ ]:
subjects = np.array(["math", "english", "science"])

col_mean = scores.mean(axis=0)
col_std = scores.std(axis=0)

for subject, mean_value, std_value in zip(subjects, col_mean, col_std):
    print(f"{subject}: 평균 {mean_value:.2f}, 표준편차 {std_value:.2f}")

best_subject = subjects[col_mean.argmax()]
print("평균 1위 과목:", best_subject)

### 왜 이 코드가 정답인지

`scores` 는 `(120, 3)` 배열이다. 행은 학생, 열은 과목이므로 과목별 평균을 구하려면 열을 유지하고 행 방향으로 계산해야 한다. NumPy에서는 이것을 `axis=0` 으로 표현한다. 결과는 과목 수와 같은 길이 3의 배열이며, `subjects` 와 같은 순서라서 `zip` 으로 함께 출력할 수 있다. `argmax` 는 가장 큰 평균의 위치를 반환하므로 같은 인덱스로 과목명을 찾는다.

**예상 핵심값**

| 과목 | 평균 | 표준편차 |
|---|---:|---:|
| math | 73.37 | 12.51 |
| english | 69.26 | 9.94 |
| science | 72.98 | 11.08 |

---

## 문제 4 정답 — 학생별 총점과 평균

In [ ]:
student_total = scores.sum(axis=1)
student_mean = scores.mean(axis=1)

print("총점 평균:", f"{student_total.mean():.2f}")
print("최고 총점:", student_total.max())
print("최저 총점:", student_total.min())

top_idx = student_total.argmax()
print("총점 1위 index:", top_idx)
print("총점 1위 반:", class_id[top_idx])
print("총점 1위 학생번호:", student_id[top_idx])
print("총점 1위 점수:", scores[top_idx])
print("총점 1위 총점:", student_total[top_idx])

print("student_total shape:", student_total.shape)
print("student_mean shape:", student_mean.shape)

### 왜 이 코드가 정답인지

학생별 총점은 한 행 안의 세 과목을 더해야 하므로 `axis=1` 이다. 결과는 학생 수만큼 값이 있는 1차원 배열 `(120,)` 이 된다. `argmax` 로 최고 총점의 인덱스를 찾으면, 같은 인덱스를 `class_id`, `student_id`, `scores` 에 적용해 해당 학생의 정보를 모두 확인할 수 있다. 여러 배열이 같은 행 순서를 공유한다는 점을 이용한 답이다.

**채점 포인트**

- `axis=1` 을 사용했는가.
- 최고 총점만 출력하지 않고 어느 반/학생인지 연결했는가.
- 총점과 평균 배열의 shape 가 120개 학생 기준인지 확인했는가.

---

## 문제 5 정답 — 반별 수학 평균 비교

In [ ]:
math_scores = scores[:, 0]

class_math_mean = np.array([
    math_scores[class_id == c].mean()
    for c in [1, 2, 3]
])

for c, mean_value in zip([1, 2, 3], class_math_mean):
    print(f"{c}반 수학 평균: {mean_value:.2f}")

best_class = class_math_mean.argmax() + 1
worst_class = class_math_mean.argmin() + 1
gap = class_math_mean.max() - class_math_mean.min()

print("수학 평균 최고 반:", best_class)
print("수학 평균 최저 반:", worst_class)
print("최고-최저 차이:", f"{gap:.2f}점")

### 왜 이 코드가 정답인지

`class_id == c` 는 학생별로 해당 반인지 아닌지를 나타내는 Boolean 배열이다. 이 Boolean 배열을 `math_scores` 에 적용하면 해당 반의 수학 점수만 남는다. 반은 1, 2, 3 세 개이므로 리스트 컴프리헨션으로 평균 세 개를 만들 수 있다. 배열의 첫 값은 1반, 두 번째 값은 2반, 세 번째 값은 3반이므로 `argmax()` 결과에 1을 더해야 실제 반 번호가 된다.

**예상 핵심값**

| 반 | 수학 평균 |
|---:|---:|
| 1반 | 73.95 |
| 2반 | 69.88 |
| 3반 | 76.28 |

---

## 문제 6 정답 — z-점수 표준화

In [ ]:
z_scores = (scores - col_mean) / col_std

print("표준화 후 평균:", np.round(z_scores.mean(axis=0), 4))
print("표준화 후 표준편차:", np.round(z_scores.std(axis=0), 4))

print("1번 학생 원점수:", scores[0])
print("1번 학생 z-점수:", np.round(z_scores[0], 3))

### 왜 이 코드가 정답인지

`col_mean` 과 `col_std` 는 길이가 3인 배열이다. `scores` 는 `(120, 3)` 이므로 NumPy는 길이 3 배열을 각 행에 맞춰 자동으로 확장한다. 이 브로드캐스팅 덕분에 모든 학생의 수학 점수에서는 수학 평균을 빼고, 영어 점수에서는 영어 평균을 빼고, 과학 점수에서는 과학 평균을 뺄 수 있다. 표준화가 맞으면 각 과목의 평균은 0 근처, 표준편차는 1 근처가 된다.

**지도 메모**

학생이 `scores.mean()` 하나로 전체 평균을 빼도 코드가 실행될 수는 있다. 하지만 과목마다 기준이 달라야 하므로 이 문제에서는 오답 처리한다. 표준화 목적은 과목별 척도를 맞추는 것이다.

---

## 문제 7 정답 — 브로드캐스팅 기본: 행마다 같은 값 더하기

In [ ]:
a = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
b = np.array([10, 20, 30])

added = a + b
multiplied = a * b
row_sum = multiplied.sum(axis=1)

print("a + b:\n", added)
print("a * b:\n", multiplied)
print("행별 합계:", row_sum)
print("a.shape:", a.shape)
print("b.shape:", b.shape)
print("(a + b).shape:", added.shape)

### 왜 이 코드가 정답인지

`a` 는 `(3, 3)` 이고 `b` 는 `(3,)` 이다. NumPy는 `b` 를 각 행에 맞춰 반복해 `a` 의 세 열에 각각 10, 20, 30을 더한다. 곱셈도 같은 방식으로 동작한다. `multiplied.sum(axis=1)` 은 곱셈 결과의 각 행을 더하므로 `[140, 320, 500]` 이 나온다. 이 문제는 브로드캐스팅이 단순 반복문 없이 열 방향 계산을 처리한다는 사실을 확인하는 문제다.

**예상 결과**

```text
a + b =
[[11 22 33]
 [14 25 36]
 [17 28 39]]
행별 합계: [140 320 500]
```

---

## 문제 8 정답 — 브로드캐스팅 응용: 열 최솟값 빼기

In [ ]:
a_min = a.min(axis=0)
a_normalized = a - a_min

print("열별 최솟값:", a_min)
print("열 최솟값 정규화:\n", a_normalized)

c = np.array([[100], [200], [300]])
print("c.shape:", c.shape)
print("a + c:\n", a + c)

### 왜 이 코드가 정답인지

`a.min(axis=0)` 은 열별 최솟값 `[1, 2, 3]` 을 만든다. 이 배열은 `(3,)` 이므로 `a - a_min` 을 하면 각 열에서 자신의 최솟값이 빠진다. 반면 `c` 는 `(3, 1)` 이다. 세 행에 각각 100, 200, 300이 대응되고, 열 방향으로 값이 반복된다. 그래서 문제 7의 `a + b` 는 열마다 다른 값을 더하고, 이 문제의 `a + c` 는 행마다 다른 값을 더한다.

**채점 포인트**

- `axis=0` 으로 열별 최솟값을 구했는가.
- `(3,)` 과 `(3, 1)` 의 결과 차이를 말로 설명할 수 있는가.

---

## 문제 9 정답 — 점수 구간별 개수 세기

In [ ]:
flat_scores = scores.reshape(-1)

bins = [
    ("60점 미만", flat_scores < 60),
    ("60~69점", (flat_scores >= 60) & (flat_scores < 70)),
    ("70~79점", (flat_scores >= 70) & (flat_scores < 80)),
    ("80~89점", (flat_scores >= 80) & (flat_scores < 90)),
    ("90점 이상", flat_scores >= 90),
]

counts = np.array([mask.sum() for _, mask in bins])

for (label, _), count in zip(bins, counts):
    ratio = count / flat_scores.size * 100
    print(f"{label}: {count}개 ({ratio:.1f}%)")

print("구간 합계:", counts.sum())
print("전체 점수 개수:", flat_scores.size)
print("가장 많은 구간:", bins[counts.argmax()][0])

### 왜 이 코드가 정답인지

`scores` 는 120명 × 3과목이라 전체 점수는 360개다. `reshape(-1)` 은 2차원 배열을 한 줄로 펼쳐 구간 조건을 쉽게 적용하게 한다. 각 구간은 Boolean 배열이며, `True` 는 1처럼 합산되므로 `.sum()` 으로 개수를 셀 수 있다. 구간을 만들 때 하한과 상한을 정확히 지정해야 점수가 중복으로 들어가지 않는다. 마지막에 구간 합계가 전체 점수 개수와 같은지 확인하면 조건 누락이나 중복을 잡을 수 있다.

**예상 핵심값**

```text
60점 미만: 54개
60~69점: 82개
70~79점: 142개
80~89점: 64개
90점 이상: 18개
```

---

## 문제 10 정답 — 주사위 CSV 분포 확인

In [ ]:
dice_raw = np.loadtxt(f"{DATA_BASE}/dice_rolls.csv", delimiter=",", skiprows=1)
total = dice_raw[:, 3].astype(int)

for s in range(2, 13):
    count = (total == s).sum()
    ratio = count / len(total) * 100
    print(f"합 {s:2d}: {count:4d}회 ({ratio:4.1f}%)")

p7 = (total == 7).mean()
print("합 7 비율:", f"{p7:.4f}")
print("합 평균:", f"{total.mean():.4f}")
print("이론 평균과 차이:", f"{abs(total.mean() - 7):.4f}")

### 왜 이 코드가 정답인지

두 주사위 합은 2부터 12까지만 가능하다. 따라서 `range(2, 13)` 으로 모든 가능한 합을 순회하고, `(total == s).sum()` 으로 각 합의 등장 횟수를 센다. 합 7은 경우의 수가 가장 많아 이론 확률이 `6/36 = 1/6` 이다. 실제 CSV는 4,000회 표본이라 정확히 1/6은 아니지만, 0.17 근처면 정상이다. 평균도 이론 평균 7 근처에 있어야 한다.

**예상 핵심값**

| 항목 | 값 |
|---|---:|
| 합 7 횟수 | 679회 |
| 합 7 비율 | 0.1698 |
| 합 평균 | 7.0380 |

---

## 문제 11 정답 — 난수 시뮬레이션으로 대수의 법칙 확인

In [ ]:
np.random.seed(2024)

for n in [10, 100, 1000, 10000]:
    die1 = np.random.randint(1, 7, n)
    die2 = np.random.randint(1, 7, n)
    simulated_total = die1 + die2

    mean_value = simulated_total.mean()
    p7_sim = (simulated_total == 7).mean()

    print(f"n={n:>5} | 평균 {mean_value:.4f} | 합 7 비율 {p7_sim:.4f}")

# 시행 횟수가 커질수록 평균은 이론값 7 근처에서 더 안정된다.

### 왜 이 코드가 정답인지

`np.random.randint(1, 7, n)` 은 1 이상 7 미만, 즉 1부터 6까지 정수를 n개 만든다. 두 주사위 배열을 더하면 시행별 합이 된다. n이 작을 때는 우연의 영향이 크지만 n이 커지면 평균이 이론값 7 주변에서 안정된다. 이것이 대수의 법칙을 코드로 확인한 것이다. 합 7 비율도 표본이 커질수록 이론 확률 1/6 근처로 이동한다.

**채점 포인트**

- `randint` 의 끝값 7이 포함되지 않는다는 점을 이해했는가.
- 매 n에서 평균과 합 7 비율을 모두 구했는가.
- 결과가 매번 조금 달라도 해석이 이론값과 연결되는가.

---

## 문제 12 정답 — 주가 데이터 기본 요약

In [ ]:
raw_stock = np.loadtxt(f"{DATA_BASE}/stock_prices.csv", delimiter=",", skiprows=1)
days = raw_stock[:, 0].astype(int)
prices = raw_stock[:, 1:]
names = np.array(["TechA", "FinB", "SmallC", "MidD", "LargeE"])

start_price = prices[0]
final_price = prices[-1]
max_price = prices.max(axis=0)
min_price = prices.min(axis=0)
price_diff = final_price - start_price

print(f"{'종목':<8} {'시작':>8} {'최종':>8} {'최고':>8} {'최저':>8} {'차이':>8}")
for i, name in enumerate(names):
    print(
        f"{name:<8} {start_price[i]:>8,.0f} {final_price[i]:>8,.0f} "
        f"{max_price[i]:>8,.0f} {min_price[i]:>8,.0f} {price_diff[i]:>8,.0f}"
    )

print("오른 종목:", names[price_diff > 0])
print("내린 종목:", names[price_diff < 0])
print("가격 차이 기준 최고 상승:", names[price_diff.argmax()])

### 왜 이 코드가 정답인지

첫 번째 열은 날짜 번호이므로 `days` 로 따로 분리하고, 나머지 5열이 종목별 가격이다. `prices[0]` 은 첫 거래일, `prices[-1]` 은 마지막 거래일이다. `max(axis=0)` 과 `min(axis=0)` 은 종목별 최고/최저 가격을 계산한다. 가격 차이 `final_price - start_price` 는 각 종목이 기간 전체에서 단순 가격 기준으로 얼마나 움직였는지 보여준다.

**예상 해석**

TechA는 시작가보다 상승했고, FinB와 LargeE는 크게 하락한다. 단순 가격 차이만으로 투자 판단을 끝내면 안 되므로 다음 문제에서 로그수익률과 변동성을 함께 본다.

---

## 문제 13 정답 — 로그수익률과 연환산 지표

In [ ]:
log_ret = np.diff(np.log(prices), axis=0)

ret_mean = log_ret.mean(axis=0)
ret_std = log_ret.std(axis=0)
ann_ret = ret_mean * 252
ann_vol = ret_std * np.sqrt(252)

print(f"{'종목':<8} {'일평균%':>10} {'일변동%':>10} {'연수익%':>10} {'연변동%':>10}")
for i, name in enumerate(names):
    print(
        f"{name:<8} {ret_mean[i] * 100:>10.3f} {ret_std[i] * 100:>10.3f} "
        f"{ann_ret[i] * 100:>10.1f} {ann_vol[i] * 100:>10.1f}"
    )

print("연환산 수익률 최고:", names[ann_ret.argmax()])
print("연환산 변동성 최고:", names[ann_vol.argmax()])

### 왜 이 코드가 정답인지

로그수익률은 `log(오늘 가격) - log(전날 가격)` 으로 계산할 수 있다. `np.diff(np.log(prices), axis=0)` 은 날짜 방향으로 인접 행 차이를 구하므로 251행 5열의 수익률 배열을 만든다. 수익률 평균에 252를 곱하면 연환산 수익률, 일간 표준편차에 `sqrt(252)` 를 곱하면 연환산 변동성이다. `argmax` 는 최고 수익률 종목과 최고 위험 종목을 찾는 데 사용한다.

**예상 핵심값**

| 항목 | 종목 |
|---|---|
| 연환산 수익률 최고 | TechA |
| 연환산 변동성 최고 | FinB |

---

## 문제 14 정답 — 센서 격자 reshape 와 축 방향 탐색

In [ ]:
sensor_raw = np.loadtxt(f"{DATA_BASE}/sensor_grid.csv", delimiter=",", skiprows=1)
temps = sensor_raw[:, 2].reshape(20, 30)

print("temps shape:", temps.shape)
print("전체 평균:", f"{temps.mean():.2f}℃")
print("최고 온도:", f"{temps.max():.1f}℃")
print("최저 온도:", f"{temps.min():.1f}℃")

max_position = tuple(map(int, np.unravel_index(temps.argmax(), temps.shape)))
print("최고 온도 위치:", max_position)

hot_col_by_row = temps.argmax(axis=1)
print("각 행의 최고 온도 열 위치 앞 5개:", hot_col_by_row[:5])

### 왜 이 코드가 정답인지

`sensor_grid.csv` 는 600개의 온도값을 가진다. 20행 30열 격자로 만들면 원소 수가 `20 * 30 = 600` 으로 맞기 때문에 `reshape(20, 30)` 이 가능하다. `temps.argmax()` 는 펼쳐진 1차원 기준 위치를 반환하므로, `np.unravel_index` 를 사용해 2차원 행/열 위치로 바꿔야 한다. `argmax(axis=1)` 은 각 행에서 열 방향으로 최고값의 위치를 찾는다.

**예상 핵심값**

| 항목 | 값 |
|---|---|
| 전체 평균 | 약 23.61℃ |
| 최고 온도 | 30.5℃ |
| 최고 온도 위치 | `(10, 16)` |

---

## 문제 15 정답 — 분석 결론 한 페이지 만들기

In [ ]:
score_best_subject = subjects[col_mean.argmax()]
dice_p7 = (total == 7).mean()
stock_best_return = names[ann_ret.argmax()]
stock_highest_risk = names[ann_vol.argmax()]
sensor_max = temps.max()
sensor_max_position = tuple(map(int, np.unravel_index(temps.argmax(), temps.shape)))

print("점수 평균 1위 과목:", score_best_subject)
print("주사위 합 7 비율:", f"{dice_p7:.4f}")
print("주가 수익률 최고 종목:", stock_best_return)
print("주가 변동성 최고 종목:", stock_highest_risk)
print("센서 최고 온도:", f"{sensor_max:.1f}℃")
print("센서 최고 온도 위치:", sensor_max_position)

### 왜 이 코드가 정답인지

마지막 문제는 새 계산보다 앞 문제의 핵심 결과를 정확히 회수하는 능력을 본다. `col_mean`, `total`, `ann_ret`, `ann_vol`, `temps` 는 모두 이전 문제에서 만든 핵심 배열이다. 이 배열에서 대표값을 다시 추출하면 결론에 들어갈 근거 숫자가 정리된다. 데이터 분석 결론은 감상이 아니라 출력값에서 확인한 수치와 항목명을 포함해야 한다.

**결론 예시**

```text
점수 데이터에서는 math 평균이 가장 높았고 english 평균이 가장 낮았다.
주사위 실험에서 합 7 비율은 약 0.17로 이론 확률 1/6과 가깝다.
주가 데이터에서는 TechA가 연환산 수익률이 가장 높았고 FinB가 변동성이 가장 컸다.
센서 격자에서는 최고 온도가 약 30.5℃였으며 위치는 10행 16열 근처였다.
```

---

## 전체 채점 메모

| 구간 | 문제 | 핵심 개념 | 필수 통과 조건 |
|---|---|---|---|
| 기본 | 1~3 | 로드, shape, axis=0 | 데이터 구조와 과목별 집계가 맞음 |
| 응용 | 4~6 | axis=1, 마스크, 표준화 | 학생별/반별 계산과 z-score가 맞음 |
| 브로드캐스팅 | 7~9 | `(3,)`, `(3,1)`, Boolean count | 결과 모양과 구간 개수 합계가 맞음 |
| 시뮬레이션 | 10~11 | 주사위 분포, 난수 | 합 7 비율과 평균 해석이 맞음 |
| 실전 | 12~15 | 주가, 센서, 결론 | 수익률/변동성/reshape/결론 근거가 맞음 |

부분 점수는 코드 실행 여부보다 계산 의도를 우선한다. 다만 `scores` 에 잘못된 열이 들어간 경우 2번 이후 점수 분석 전체가 왜곡되므로 문제 2를 먼저 고치게 한다. 주가 분석에서 단순 가격 차이와 로그수익률을 혼동한 학생은 문제 12는 부분 통과, 문제 13은 재제출로 처리한다.

## 학생 답안에서 자주 보는 패턴

| 패턴 | 의미 | 교사 피드백 |
|---|---|---|
| `raw[2:]` | 행 슬라이싱과 열 슬라이싱 혼동 | `raw[:, 2:]` 에서 쉼표가 행/열 구분임을 설명 |
| `axis` 생략 | 전체 평균만 계산 | 배열 모양을 먼저 쓰고 어느 방향으로 줄일지 묻기 |
| `class_id = raw[0]` | 첫 행을 반 번호로 착각 | `raw[:, 0]` 과 `raw[0, :]` 를 비교 시연 |
| `reshape(30, 20)` | 격자 방향 임의 변경 | 데이터 설명의 20행 30열 구조를 다시 확인 |
| `price_diff` 만 보고 결론 | 수익률/위험 누락 | 가격 차이와 수익률은 다른 질문이라고 안내 |
| 결론에 숫자 없음 | 계산 결과와 글쓰기 연결 실패 | 적어도 비율, 종목명, 온도 중 2개 이상 넣게 함 |

## 재실행 확인 순서

1. 런타임을 새로 시작한다.
2. 환경 셀부터 문제 15까지 순서대로 실행한다.
3. 출력이 없는 셀은 있어도 되지만, 오류가 나는 셀은 없어야 한다.
4. `scores.shape`, `prices.shape`, `temps.shape` 가 각각 `(120, 3)`, `(252, 5)`, `(20, 30)` 인지 확인한다.
5. 학생 결론 셀에 코드 출력과 같은 항목명이 들어 있는지 확인한다.

이 레슨은 NumPy의 핵심인 "배열 모양을 먼저 보고, 축 방향을 정한 뒤, 반복문 대신 배열 연산으로 계산한다"는 습관을 만드는 수업이다. 결과 숫자가 조금 다르게 보이면 먼저 데이터 로드와 슬라이싱을 확인하고, 그다음 axis 방향을 확인한다.

## 문제별 지도 질문

| 문제 | 학생에게 던질 질문 | 확인할 답 |
|---:|---|---|
| 1 | `shape` 와 `size` 는 각각 무엇을 말하나요? | 행/열 구조와 전체 원소 수를 구분 |
| 2 | `raw[:, 2:]` 에서 첫 번째 `:` 는 무엇인가요? | 모든 행 선택 |
| 3 | 과목별 평균에서 왜 `axis=0` 을 쓰나요? | 행 방향으로 줄여 열별 결과를 남김 |
| 4 | 총점 1위 학생을 찾을 때 왜 같은 index를 여러 배열에 쓰나요? | 배열들이 같은 행 순서를 공유함 |
| 5 | `class_id == 2` 의 결과 타입은 무엇인가요? | Boolean 배열 |
| 6 | z-점수의 평균과 표준편차는 어떤 값 근처여야 하나요? | 평균 0, 표준편차 1 |
| 7 | `(3,)` 배열은 어느 방향으로 반복되나요? | 각 행에 같은 열 보정값 적용 |
| 8 | `(3, 1)` 배열은 문제 7과 무엇이 다른가요? | 행마다 다른 값이 열 방향으로 반복 |
| 9 | 구간 조건이 겹치면 어떤 문제가 생기나요? | 같은 점수가 여러 구간에 중복 집계 |
| 10 | 합 7이 가장 많은 이유는 무엇인가요? | 가능한 조합 수가 6개로 가장 많음 |
| 11 | 시행 횟수가 늘어날수록 어떤 값이 안정되나요? | 평균과 합 7 비율 |
| 12 | 가격 차이만으로 투자 판단을 끝내면 왜 부족한가요? | 위험과 수익률을 반영하지 않음 |
| 13 | 변동성 최고와 수익률 최고가 다른 의미인 이유는 무엇인가요? | 위험과 기대 성과가 다른 지표 |
| 14 | `argmax()` 결과를 왜 `unravel_index` 로 바꾸나요? | 1차원 위치를 2차원 행/열로 변환 |
| 15 | 결론이 좋은지 확인하는 가장 빠른 방법은 무엇인가요? | 코드 출력 숫자와 항목명이 포함됨 |

## 보충 설명 포인트

- `axis=0` 과 `axis=1` 은 학생들이 가장 많이 헷갈린다. 칠판에 `(120, 3)` 을 적고 "어느 방향이 사라지는가"로 설명하면 이해가 빠르다.
- 브로드캐스팅은 규칙 암기보다 결과 모양 예측이 중요하다. 계산 전에 `a.shape`, `b.shape`, `c.shape` 를 먼저 출력하게 한다.
- 난수 시뮬레이션은 정답 숫자가 완전히 고정되는 문제와 다르다. seed 를 고정한 경우의 재현성과 표본이 커질수록 안정되는 통계적 성질을 분리해 설명한다.
- 주가 문제는 투자 조언이 목적이 아니라 배열 기반 시계열 요약이 목적이다. 학생이 "이 종목을 사야 한다"로만 결론을 쓰면 수익률, 변동성, 기간이라는 근거를 다시 요구한다.
- 센서 문제는 2차원 배열의 실전 감각을 만드는 문제다. `reshape` 이후 행/열 의미가 바뀌면 위치 해석도 틀어진다는 점을 강조한다.